In [104]:
import numpy as np
import pandas as pd

In [105]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

In [106]:
df = pd.read_csv('covid_toy.csv')

In [107]:
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [108]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [109]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [110]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['has_covid']),df['has_covid'],test_size=0.2)

In [111]:
X_train

,age,gender,fever,cough,city
89,46,Male,103.0,Strong,Bangalore
1,27,Male,100.0,Mild,Delhi
76,80,Male,100.0,Mild,Bangalore
73,34,Male,98.0,Strong,Kolkata
68,54,Female,104.0,Strong,Kolkata
...,...,...,...,...,...
98,5,Female,98.0,Strong,Mumbai
8,19,Female,100.0,Strong,Bangalore
6,14,Male,101.0,Strong,Bangalore
96,51,Female,101.0,Strong,Kolkata


# method 1

In [112]:
# adding simple imputer to fever column

si = SimpleImputer()
X_train_fever = si.fit_transform(X_train[['fever']])

In [113]:
# also the test data
X_test_fever = si.fit_transform(X_test[['fever']])


In [114]:
X_train_fever.shape

(80, 1)

In [115]:
# ordinalencoding -> cough
oe = OrdinalEncoder(categories =[['Mild','Strong']])
X_train_cough = oe.fit_transform(X_train[['cough']])

# also the test data

X_test_cough = oe.fit_transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [116]:
# onehotencoding -> gender,city

ohe = OneHotEncoder(drop='first',sparse_output = False)
X_train_gender_city = ohe.fit_transform(X_train[['gender','city']])

# also the test data

X_test_gender_city = ohe.fit_transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [117]:
# extracting age

X_train_age = X_train.drop(columns=['gender','fever','cough','city']).values

# also the test data
X_test_age = X_test.drop(columns=['gender','fever','cough','city']).values
X_train_age.shape

(80, 1)

In [118]:
print(type(X_train_age), getattr(X_train_age, 'shape', None))
print(type(X_train_fever), getattr(X_train_fever, 'shape', None))
print(type(X_train_gender_city), getattr(X_train_gender_city, 'shape', None))
print(type(X_train_cough), getattr(X_train_cough, 'shape', None))

<class 'numpy.ndarray'> (80, 1)
<class 'numpy.ndarray'> (80, 1)
<class 'numpy.ndarray'> (80, 4)
<class 'numpy.ndarray'> (80, 1)


In [119]:

X_train_transformed = np.concatenate((X_train_age,X_train_fever,X_train_gender_city,X_train_cough),axis=1)
# also the test data
X_test_transformed = np.concatenate((X_test_age,X_test_fever,X_test_gender_city,X_test_cough),axis=1)

X_train_transformed.shape

(80, 7)

# method-2


In [120]:
from sklearn.compose import ColumnTransformer

In [121]:
transformer = ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(sparse_output=False,drop='first'),['gender','city'])
],remainder='passthrough')

In [122]:
transformer.fit_transform(X_train).shape

(80, 7)

In [123]:

transformer.transform(X_test).shape


(20, 7)